In [1]:
%env CUDA_VISIBLE_DEVICES=GPU-8868e167-e666-53c7-6c41-d8e83081f07e

env: CUDA_VISIBLE_DEVICES=GPU-8868e167-e666-53c7-6c41-d8e83081f07e


In [2]:
import pandas as pd

#load data
data = pd.read_csv(r"/home/lero/idrive/cmac/DDMAP/Stability studies/Stability_dataset_August_update.csv", na_values='nan')
data = data.drop(data.columns[0:7], axis=1)
data.drop(['Unnamed: 209', 'Unnamed: 210', 'Unnamed: 0'], axis=1, inplace=True)

#data = pd.read_csv(r"/home/lero/idrive/cmac/DDMAP/Stability studies/ML_test_set.csv", na_values='nan')

#Store values for API/ polymer, condition
original_api = data['API']
original_polymer = data['Polymer']
original_condition = data['condition']

#fill pure api with 0 for polymer mol desc
pure = data['Polymer']=='Pure'
polymer_descriptors = data.columns[219:]
data.loc[pure, polymer_descriptors] = 0

#drop conditions since these have been split into temp/ humidity
data.drop(['condition'], axis=1, inplace=True)

#fill na values with average
data.fillna(data.mean(numeric_only=True), inplace=True)

#Define Features for ColumnTransformer (AFTER ALL DROPS within dataframe) ---
categorical_features = ['API', 'Polymer']
# Identify numerical features:
dont_scale_features = data.drop(['Average days stable', 'GFA'], axis=1).columns.tolist()
numerical_features = [col for col in dont_scale_features if col not in categorical_features]

data.drop(['API', 'Polymer'], inplace=True, axis=1)

data.head()
#print(unique_apis)

,Average days stable,Drug loading (wt%),GFA,Tm (°C),Tg (°C),ΔHfus (kJ mol–1),ΔSfus × 10–2 (kJ mol–1 K–1),MaxAbsEStateIndex_x,MaxEStateIndex_x,MinAbsEStateIndex_x,...,fr_sulfide_y,fr_sulfonamd_y,fr_sulfone_y,fr_term_acetylene_y,fr_tetrazole_y,fr_thiazole_y,fr_thiocyan_y,fr_thiophene_y,fr_unbrch_alkane_y,fr_urea_y
0,1080.000000,80,2.0,166.0,25.2,39.272748,7.892092,10.524469,10.524469,0.115102,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1081.225694,70,2.0,166.0,25.2,39.272748,7.892092,10.524469,10.524469,0.115102,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1081.225694,60,2.0,166.0,25.2,39.272748,7.892092,10.524469,10.524469,0.115102,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2160.000000,50,2.0,166.0,25.2,39.272748,7.892092,10.524469,10.524469,0.115102,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2160.000000,40,2.0,166.0,25.2,39.272748,7.892092,10.524469,10.524469,0.115102,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
#split data
X = data.drop(['Average days stable'], axis=1)
y_non_binary = data['Average days stable']
y = (y_non_binary>=2160).astype(int)

In [4]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

models = {
    'Logistic Regression': (
        LogisticRegression(max_iter=1000000),
        [
            {  
                'model__C': np.logspace(-4, 4, 20), 
                'model__l1_ratio': [0],
                'model__solver': ['lbfgs', 'newton-cg', 'sag']
            },
            {
                'model__C': np.logspace(-4, 4, 20),
                'model__solver': ['liblinear'],
                'model__l1_ratio': [1],
            },
            {
                'model__C': np.logspace(-4, 4, 20),
                'model__penalty': ['elasticnet'],
                'model__solver': ['saga'],
                'model__l1_ratio': [0.1, 0.5, 0.9]
            }
        ]
    ),
    # 'Linear SVC': (
    #     LinearSVC(max_iter=100000),
    #     {
    #         'model__C': [0.01, 0.1, 1, 10, 100]  
    #     }
    # ),
    # 'K Neighbors Classifier': (
    #     KNeighborsClassifier(), 
    #     {
    #         'model__n_neighbors': np.arange(2,30,1)
    #     }
    # ),
    'Random Forest Classifier': (
        RandomForestClassifier(random_state=42), 
        {
            'model__n_estimators': [300, 500, 1000, 1500, 2000, 5000],
            'model__max_features': ['sqrt', 'log2', None],
            'model__max_depth': [None, 10, 50, 100, 300],
            'model__min_samples_split': [2, 5, 10, 20, 50]
        }
    ),
    'XGBoost classifier': (
        XGBClassifier(random_state=42),
        {
            'model__max_depth': [3, 10, 50, 100, 300],
            'model__subsample': [0.5, 0.7, 0.9, 1],
            'model__colsample_bytree': [0.5, 0.7, 0.9, 1],
            'model__learning_rate': [0.001, 0.01, 0.05, 0.1],
            'model__n_estimators': [500, 1000, 1500, 2000],
        }
    ),
    'MLP Classifier': (
        MLPClassifier(max_iter=100000000, early_stopping=True), 
        {
            'model__hidden_layer_sizes': [(100,), (200,), (100, 50), (200, 200)],
            'model__activation': ['logistic', 'tanh', 'relu'],
            'model__alpha': [0.0001, 0.001, 0.01, 0.05, 0.1],
            'model__solver': ['sgd', 'adam'],
            'model__learning_rate': ['constant', 'invscaling', 'adaptive'],
            'model__learning_rate_init': [0.001, 0.01, 0.1],
        }
    )
}

In [5]:
#pre-processor for one-hot encoding and scaling
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer(
    transformers=[ 
        ('num', StandardScaler(), numerical_features),
    ],
    remainder = 'passthrough' # Keep any other columns not explicitly transformed (e.g., if there are any not in num or cat)
)

In [ ]:
#Nested cv
from sklearn.metrics import make_scorer, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, cross_val_score, GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
import pickle
from sklearn.metrics import f1_score
import os
from tqdm.auto import tqdm

results = {}
scorer = make_scorer(f1_score, average='binary')

#groups for GroupKFold
groups = (original_api.astype(str)).values

#GroupKFold for outer cv
outer_cv = GroupKFold(n_splits=5) #change n_splits to 80:20
inner_cv = GroupKFold(n_splits=5) #change n_splits to 80:20

#directory to save the models
save_directory = '/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results/Classifier/January_re-run'
os.makedirs(save_directory, exist_ok=True)

for model_name, (classifier, param_grid) in tqdm(models.items(), desc='models', total=len(models)):
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', classifier)
    ])
    
    print('Model:', model_name)
  
    # Perform nested cross-validation
    grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=inner_cv, scoring=scorer, verbose=10, n_jobs=100)
    
    fit_params = {'groups': groups}
    
    # Evaluate outer loop scores
    nested_score = cross_val_score(grid_search, X, y, groups=groups, cv=outer_cv, params=fit_params, n_jobs=1)
    
    # Get predictions
    predictions = cross_val_predict(grid_search, X, y, groups=groups, cv=outer_cv, params=fit_params, method='predict', n_jobs=1)

    # Fit to find best parameters
    grid_search.fit(X, y, **fit_params)
    best_params = grid_search.best_params_
    
    # Save the best model
    best_model = grid_search.best_estimator_
    model_file_path = os.path.join(save_directory, f'{model_name}_best_model.pkl')
    with open(model_file_path, 'wb') as model_file:
        pickle.dump(best_model, model_file)

    results[model_name] = {
        'nested_score': nested_score,
        'ground_truth': y.values,
        'predictions': predictions,
        'best_params': best_params
    }

dictionary_file_path = os.path.join(save_directory, 'Classifiers_results_dictionary.pkl')   
with open(dictionary_file_path, 'wb') as f:
    pickle.dump(results, f)
    
print('Finito')


models:   0%|          | 0/4 [00:00<?, ?it/s]

Model: Logistic Regression
Fitting 5 folds for each of 140 candidates, totalling 700 fits


/home/lero/.conda/envs/lero_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/lero/.conda/envs/lero_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/lero/.conda/envs/lero_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be re

Fitting 5 folds for each of 140 candidates, totalling 700 fits


/home/lero/.conda/envs/lero_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/lero/.conda/envs/lero_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/lero/.conda/envs/lero_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be re

...


[CV 5/5; 111/140] END model__C=1.623776739188721, model__l1_ratio=0.1, model__penalty=elasticnet, model__solver=saga;, score=0.592 total time=   2.4s
[CV 1/5; 112/140] START model__C=1.623776739188721, model__l1_ratio=0.5, model__penalty=elasticnet, model__solver=saga
[CV 1/5; 112/140] END model__C=1.623776739188721, model__l1_ratio=0.5, model__penalty=elasticnet, model__solver=saga;, score=0.687 total time=   8.4s
[CV 5/5; 15/140] START model__C=0.004832930238571752, model__l1_ratio=0, model__solver=sag
[CV 5/5; 15/140] END model__C=0.004832930238571752, model__l1_ratio=0, model__solver=sag;, score=0.795 total time=   0.5s
[CV 3/5; 43/140] START model__C=78.47599703514607, model__l1_ratio=0, model__solver=lbfgs
[CV 3/5; 43/140] END model__C=78.47599703514607, model__l1_ratio=0, model__solver=lbfgs;, score=0.762 total time=   1.0s
[CV 4/5; 43/140] START model__C=78.47599703514607, model__l1_ratio=0, model__solver=lbfgs
[CV 4/5; 43/140] END model__C=78.47599703514607, model__l1_ratio=0

/home/lero/.conda/envs/lero_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Model: Random Forest Classifier
Fitting 5 folds for each of 450 candidates, totalling 2250 fits

[CV 4/5; 74/140] START model__C=29.763514416313132, model__l1_ratio=1, model__solver=liblinear
[CV 4/5; 74/140] END model__C=29.763514416313132, model__l1_ratio=1, model__solver=liblinear;, score=0.519 total time=   1.5s
[CV 5/5; 74/140] START model__C=29.763514416313132, model__l1_ratio=1, model__solver=liblinear
[CV 5/5; 74/140] END model__C=29.763514416313132, model__l1_ratio=1, model__solver=liblinear;, score=0.800 total time=   1.5s
[CV 1/5; 126/140] START model__C=206.913808111479, model__l1_ratio=0.1, model__penalty=elasticnet, model__solver=saga
[CV 1/5; 126/140] END model__C=206.913808111479, model__l1_ratio=0.1, model__penalty=elasticnet, model__solver=saga;, score=0.682 total time=   5.1s
[CV 1/5; 11/140] START model__C=0.0018329807108324356, model__l1_ratio=0, model__solver=newton-cg
[CV 1/5; 11/140] END model__C=0.0018329807108324356, model__l1_ratio=0, model__solver=newton-cg;

/home/lero/.conda/envs/lero_env/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



[CV 3/5; 137/140] START model__C=3792.690190732246, model__l1_ratio=0.9, model__penalty=elasticnet, model__solver=saga
[CV 3/5; 137/140] END model__C=3792.690190732246, model__l1_ratio=0.9, model__penalty=elasticnet, model__solver=saga;, score=0.754 total time=   2.2s
[CV 4/5; 9/140] START model__C=0.0006951927961775605, model__l1_ratio=0, model__solver=sag
[CV 4/5; 9/140] END model__C=0.0006951927961775605, model__l1_ratio=0, model__solver=sag;, score=0.508 total time=   0.2s
[CV 5/5; 28/140] START model__C=0.615848211066026, model__l1_ratio=0, model__solver=lbfgs
[CV 5/5; 28/140] END model__C=0.615848211066026, model__l1_ratio=0, model__solver=lbfgs;, score=0.633 total time=   0.4s
[CV 2/5; 50/140] START model__C=545.5594781168514, model__l1_ratio=0, model__solver=newton-cg
[CV 2/5; 50/140] END model__C=545.5594781168514, model__l1_ratio=0, model__solver=newton-cg;, score=0.789 total time=   0.3s
[CV 3/5; 50/140] START model__C=545.5594781168514, model__l1_ratio=0, model__solver=new

In [ ]:
# + tags=[]
#model scoring
import pickle
import pandas as pd
import numpy as np
import os

with open('/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results/Classifier/July_api_no/Classifiers_results_dictionary.pkl', 'rb') as f:
    results = pickle.load(f)
    
records = []

for model in results:
    score = results[model]['nested_score']
    mean_score = np.mean(score)
    records.append({'Model': model, 'Score': mean_score})
    
results_df = pd.DataFrame(records)
results_pivot = results_df.pivot(columns='Model', values='Score')
results_df

In [ ]:
#Visualise model scores
from sklearn.metrics import classification_report
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# List to hold each report entry as a dictionary
reports_list = []

for model in results:
    # Obtain the classification report as a dictionary
    report = classification_report(results[model]['ground_truth'], 
                                   results[model]['predictions'], 
                                   output_dict=True, 
                                   digits=2)
    
    # Flatten the dictionary and add to reports_list
    for class_label, metrics in report.items():
        if isinstance(metrics, dict):  # Ignore the 'accuracy' mean metrics lines
            for metric_name, metric_value in metrics.items():
                reports_list.append({
                    "Model": model,
                    "Class": class_label,
                    "Metric": metric_name,
                    "Value": metric_value
                })

# Convert the list of dictionaries into a DataFrame
reports_df = pd.DataFrame(reports_list)

# Create pivot table to organize data better
pivot_table = reports_df.pivot_table(index=['Model', 'Class'], 
                                     columns='Metric', 
                                     values='Value')

pivot_table.drop(columns='support', inplace=True)

plt.figure(figsize = (8,6), dpi=500)
sns.heatmap(pivot_table, annot = True)
plt.title('Classification Report Metrics Heatmap')
plt.xlabel('Metrics')
plt.xticks(rotation =45)
plt.ylabel('Model and Class')
plt.tight_layout()
#save_directory = '/projects/cp/se_users/ksrn200/Classifier/GroupKFold/Plots/One_week_stability'
#plot_filename = os.path.join(save_directory, 'classification_report.png')
#plt.savefig(plot_filename)

plt.show()


# Display the pivot table
print(pivot_table)

In [ ]:
#visualisation of model performance
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
import os


for model in results:
    print(model)
    cf_matrix = confusion_matrix(results[model]['ground_truth'], results[model]['predictions'])
    print(cf_matrix)
    print(classification_report(results[model]['ground_truth'], results[model]['predictions'], digits=2))
    plt.figure(figsize=(5,5), dpi=500)
    plt.title(f'{model}: results')
    sns.heatmap(normalize(cf_matrix, axis=1, norm='l1'), annot=True, fmt='.2%', cmap='Blues')
    plt.xlabel('predicted values')
    plt.ylabel('actual values')
    plt.xticks(ticks=[0.5, 1.5], labels=['0', '1'], fontsize=10, rotation=0)
    plt.yticks(ticks=[0.5, 1.5], labels=['0', '1'], fontsize=10, rotation=0)
    
    save_directory = '/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results'
    plot_filename = os.path.join(save_directory, f'{model}_results.png')
    plt.savefig(plot_filename)
    
    plt.show()


In [ ]:
#parameters of importance
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the model pipeline
with open('/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results/Classifier/July_api_no/Random Forest Classifier_best_model.pkl', 'rb') as f:
    pipeline = pickle.load(f)
    
# Transform the features
X_transformed = pipeline.named_steps['preprocessor'].transform(X)

model = pipeline.named_steps['model']

importances = model.feature_importances_

feature_names = X.columns

importance_df = pd.DataFrame({'feature': feature_names, 'Importance': importances})
importance_df.sort_values(by = 'Importance', inplace=True, ascending=False)
top_parameters = importance_df.iloc[:10]
print(top_parameters)

plt.figure(figsize=(10,7), dpi=500)
sns.barplot(top_parameters, x='feature', y='Importance')
plt.title('Feature importance using Random Forest classifier')
plt.xticks(rotation =90)
plt.xlabel('Features')
plt.ylabel('Mean accuracy decrease')
plt.tight_layout()
save_directory = '/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results'
plot_filename = os.path.join(save_directory, 'Randon_forest_classifier_feature_importance.png')
plt.savefig(plot_filename)
plt.show()
